# CFH — Canal Vocal: eGeMAPS con OpenSMILE
**Capa 3 — Indicadores vocales (y₁₂)**
Extrae 88 features acústicos eGeMAPS v02 por segmento diarizado del compareciente.
Subcasos: Casanare, Catatumbo, Dabeiba, Huila (Costa Caribe = DRM, skip)


In [ ]:
# ── 1. Instalar dependencias ──────────────────────────────────────────
!pip install opensmile soundfile librosa -q
!pip install pyannote.audio -q
print('✓ Dependencias instaladas')

In [ ]:
# ── 2. Montar Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
VIDEO_DIR = '/content/drive/MyDrive/CFH_videos'
WAV_DIR   = '/content/drive/MyDrive/CFH_wavs'
OUT_DIR   = '/content/drive/MyDrive/CFH_egemap_features'
os.makedirs(WAV_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print('✓ Drive montado')
print('Videos disponibles:', os.listdir(VIDEO_DIR) if os.path.exists(VIDEO_DIR) else 'NO ENCONTRADO')

In [ ]:
# ── 3. Extraer WAVs de los MP4 ────────────────────────────────────────
import subprocess
from pathlib import Path

subcasos = {
    'casanare':     'casanare_torres_escalante.mp4',
    'catatumbo':    'catatumbo_chaparro.mp4',
    'dabeiba':      'dabeiba_audiencia.mp4',
    'huila':        'huila_ollo_latapia.mp4',
}

for subcaso, mp4_name in subcasos.items():
    mp4_path = f'{VIDEO_DIR}/{mp4_name}'
    wav_path = f'{WAV_DIR}/{subcaso}_16k.wav'
    
    if not os.path.exists(mp4_path):
        # Buscar con nombre aproximado
        matches = [f for f in os.listdir(VIDEO_DIR) if subcaso.lower() in f.lower()]
        if matches:
            mp4_path = f'{VIDEO_DIR}/{matches[0]}'
            print(f'  Encontrado: {matches[0]}')
        else:
            print(f'  ⚠️  No encontrado: {subcaso}')
            continue
    
    if os.path.exists(wav_path):
        print(f'  ✓ WAV ya existe: {subcaso}')
        continue
    
    print(f'  Extrayendo WAV: {subcaso}...')
    cmd = f'ffmpeg -i "{mp4_path}" -ar 16000 -ac 1 -vn "{wav_path}" -y -loglevel error'
    r = subprocess.run(cmd, shell=True)
    if r.returncode == 0:
        size = os.path.getsize(wav_path) // (1024*1024)
        print(f'  ✓ {wav_path} ({size} MB)')
    else:
        print(f'  ❌ Error en {subcaso}')

In [ ]:
# ── 4. Cargar diarizaciones del compareciente ─────────────────────────
# Las diarizaciones están en el repo — cargar CSVs de AUs
# Los segmentos del compareciente están identificados como SPEAKER_03 (Casanare)
# SPEAKER_03 (Catatumbo), SPEAKER_01 (Dabeiba), SPEAKER_01/02 (Huila)

# Segmentos de compareciente por subcaso (extraídos de la diarización)
SPEAKER_SEGMENTS = {
    'casanare': 'SPEAKER_03',    # General Torres Escalante
    'catatumbo': 'SPEAKER_03',   # Capitán Chaparro  
    'dabeiba': 'SPEAKER_01',     # Oficial coronel/teniente coronel
    'huila': ['SPEAKER_01', 'SPEAKER_02'],  # Ollo La Tapia + otros
}
print('✓ Segmentos de compareciente definidos')

In [ ]:
# ── 5. Extraer features eGeMAPS con opensmile ────────────────────────
import opensmile
import soundfile as sf
import numpy as np
import pandas as pd
from pathlib import Path

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)
print(f'✓ OpenSMILE inicializado — {len(smile.feature_names)} features eGeMAPS v02')
print('Features:', smile.feature_names[:10], '...')

In [ ]:
# ── 6. Procesar cada subcaso ──────────────────────────────────────────
import librosa
import json

# Cargar diarización (RTTM o JSON de pyannote)
def cargar_segmentos_rttm(rttm_path, speaker_id):
    """Carga segmentos de un speaker desde archivo RTTM."""
    segmentos = []
    with open(rttm_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 8 and parts[7] == speaker_id:
                onset = float(parts[3])
                duration = float(parts[4])
                segmentos.append((onset, onset + duration))
    return segmentos

def extraer_egemap_segmentos(wav_path, segmentos, smile, min_dur=1.0):
    """Extrae features eGeMAPS de cada segmento de audio."""
    audio, sr = librosa.load(wav_path, sr=16000)
    resultados = []
    
    for i, (start, end) in enumerate(segmentos):
        dur = end - start
        if dur < min_dur:  # Segmentos muy cortos no son confiables
            continue
        
        start_sample = int(start * sr)
        end_sample = int(end * sr)
        segment_audio = audio[start_sample:end_sample]
        
        try:
            features = smile.process_signal(segment_audio, sr)
            feat_dict = features.iloc[0].to_dict()
            feat_dict['segment_id'] = i
            feat_dict['start_s'] = start
            feat_dict['end_s'] = end
            feat_dict['duration_s'] = dur
            resultados.append(feat_dict)
        except Exception as e:
            print(f'    Segmento {i} error: {e}')
    
    return resultados

todos_resultados = {}

for subcaso, speaker in SPEAKER_SEGMENTS.items():
    wav_path = f'{WAV_DIR}/{subcaso}_16k.wav'
    if not os.path.exists(wav_path):
        print(f'⚠️  WAV no encontrado: {subcaso}')
        continue
    
    # Buscar archivo RTTM de diarización
    rttm_candidates = [
        f'/content/drive/MyDrive/CFH_diarization/{subcaso}.rttm',
        f'{VIDEO_DIR}/{subcaso}.rttm',
    ]
    rttm_path = next((p for p in rttm_candidates if os.path.exists(p)), None)
    
    speakers = [speaker] if isinstance(speaker, str) else speaker
    
    print(f'\n{subcaso.upper()} — Speaker(s): {speakers}')
    
    if rttm_path:
        # Usar diarización existente
        segmentos = []
        for spk in speakers:
            segmentos.extend(cargar_segmentos_rttm(rttm_path, spk))
        segmentos.sort(key=lambda x: x[0])
        print(f'  Segmentos desde RTTM: {len(segmentos)}')
    else:
        # Sin diarización — usar ventanas deslizantes de 5 segundos
        print(f'  ⚠️  Sin RTTM — usando ventanas de 5s sobre audio completo')
        audio, sr = librosa.load(wav_path, sr=16000)
        dur_total = len(audio) / sr
        segmentos = [(i*5, min((i+1)*5, dur_total)) for i in range(int(dur_total/5))]
        print(f'  Segmentos (ventana 5s): {len(segmentos)}')
    
    resultados = extraer_egemap_segmentos(wav_path, segmentos, smile)
    todos_resultados[subcaso] = resultados
    
    # Estadísticas clave
    if resultados:
        df_sub = pd.DataFrame(resultados)
        print(f'  ✓ {len(resultados)} segmentos procesados')
        print(f'  F0 semitone: media={df_sub["F0semitoneFrom27.5Hz_sma3nz_amean"].mean():.2f}')
        print(f'  Shimmer:     media={df_sub["shimmerLocaldB_sma3nz_amean"].mean():.4f}')
        print(f'  Jitter:      media={df_sub["jitterLocal_sma3nz_amean"].mean():.4f}')
        print(f'  HNR:         media={df_sub["HNRdBACF_sma3nz_amean"].mean():.2f}')

In [ ]:
# ── 7. Calcular ICM vocal por subcaso ─────────────────────────────────
# El ICM vocal se basa en shimmer (quiebre de voz) y pausas
# Alta shimmer + largas pausas = mayor distress vocal = mayor congruencia ICM

def calcular_icm_vocal(df_segmentos):
    """Calcula score ICM vocal a partir de features eGeMAPS."""
    # Normalizar features clave
    shimmer = df_segmentos['shimmerLocaldB_sma3nz_amean']
    jitter  = df_segmentos['jitterLocal_sma3nz_amean']
    hnr     = df_segmentos['HNRdBACF_sma3nz_amean']
    f0_mean = df_segmentos['F0semitoneFrom27.5Hz_sma3nz_amean']
    f0_std  = df_segmentos['F0semitoneFrom27.5Hz_sma3nz_stddevNorm']
    
    # ICM vocal = (shimmer_norm + f0_std_norm) / 2
    # Mayor shimmer y variabilidad F0 = mayor expresividad emocional vocal
    def norm01(s): return (s - s.min()) / (s.max() - s.min() + 1e-9)
    
    icm_vocal = (norm01(shimmer) + norm01(f0_std)) / 2
    return icm_vocal.mean()

print('\n── ICM Vocal por subcaso ──')
icm_vocal_results = {}
for subcaso, resultados in todos_resultados.items():
    if not resultados:
        continue
    df = pd.DataFrame(resultados)
    icm_v = calcular_icm_vocal(df)
    icm_vocal_results[subcaso] = icm_v
    print(f'  {subcaso:15}: ICM_vocal = {icm_v:.3f}')

In [ ]:
# ── 8. Guardar resultados ────────────────────────────────────────────
import json

# Guardar CSV por subcaso
for subcaso, resultados in todos_resultados.items():
    if resultados:
        df = pd.DataFrame(resultados)
        out_path = f'{OUT_DIR}/egemap_{subcaso}.csv'
        df.to_csv(out_path, index=False)
        print(f'✓ {out_path} ({len(df)} segmentos, {len(df.columns)} features)')

# Guardar resumen ICM vocal
resumen = {
    'subcasos': {
        subcaso: {
            'icm_vocal': float(icm_vocal_results.get(subcaso, 0)),
            'n_segmentos': len(todos_resultados.get(subcaso, [])),
        }
        for subcaso in ['casanare','catatumbo','dabeiba','huila']
    }
}
with open(f'{OUT_DIR}/icm_vocal_resumen.json','w') as f:
    json.dump(resumen, f, indent=2)
print('\n✓ icm_vocal_resumen.json guardado')
print(json.dumps(resumen, indent=2))

In [ ]:
# ── 9. Comparar ICM facial vs. vocal ─────────────────────────────────
import pandas as pd

icm_facial = {
    'casanare':  0.190,
    'catatumbo': 0.272,
    'dabeiba':   0.353,
    'huila':     0.299,
}

print('\n── Comparación ICM facial vs. vocal ──')
print(f'{"Subcaso":15} {"ICM_facial":12} {"ICM_vocal":12} {"Convergencia":12}')
print('-'*55)
for sc in ['casanare','catatumbo','dabeiba','huila']:
    facial = icm_facial.get(sc, 0)
    vocal  = icm_vocal_results.get(sc, 0)
    conv   = abs(facial - vocal)  # Baja diferencia = alta convergencia
    print(f'{sc:15} {facial:12.3f} {vocal:12.3f} {conv:12.3f}')